# Databricks Auto Loader

**Auto Loader** is a Databricks feature used to **automatically and incrementally ingest new files** from cloud storage such as **GCS, S3, or ADLS**.

### How it works

```text
Cloud Storage
     ↓
 Auto Loader
     ↓
   Bronze
     ↓
  Silver
     ↓
   Gold
```

Instead of repeatedly scanning the entire folder, Auto Loader **discovers new files and processes them incrementally**.

### Example

```python
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .load("/raw/orders/")
)
```

### Auto Loader vs UPSERT

| Feature            | Purpose                    |
| ------------------ | -------------------------- |
| **Auto Loader**    | Ingests new files          |
| **UPSERT / MERGE** | Inserts or updates records |
| **Delta Lake**     | Stores and manages data    |

> **Key idea:** Auto Loader handles **file ingestion**, while UPSERT handles **record-level changes**.



### Auto Loader: Schema Checkpoint & Schema Location

- **cloudFiles.schemaLocation**: Path where Auto Loader stores schema checkpoints and metadata for tracking schema changes.
- **cloudFiles.schemaCheckpointLocation**: (Deprecated) Older option for schema checkpoint; use `cloudFiles.schemaLocation` instead.

### Schema Evolution Modes

| Mode      | Definition                                      |
|-----------|-------------------------------------------------|
| **addNewColumns** | Adds new columns to schema automatically. |
| **rescue**        | Captures unexpected columns in a `_rescued_data` column. |
| **none**          | No schema evolution; errors on schema mismatch. |

> Use `cloudFiles.schemaEvolutionMode` to control how Auto Loader handles schema changes.

In [0]:
df = spark.readStream\
    .format("cloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.schemaLocation",
            "/Volumes/data_engineer/bronze/autovol/schema/")\
    .option("cloudFiles.schemaEvolutionMode", "rescue")\
    .load("/Volumes/data_engineer/bronze/autovol/raw/")

In [0]:
df.writeStream\
        .format("delta")\
        .outputMode("append")\
        .option(\
            "checkpointLocation",\
            "/Volumes/data_engineer/bronze/autovol/destination/checkpoint/"\
        )\
        .trigger(once=True)\
        .start("/Volumes/data_engineer/bronze/autovol/destination/data/")

In [0]:
df = spark.read.format("delta").load("/Volumes/data_engineer/bronze/autovol/destination/data/")
display(df)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
rescued_schema = StructType()\
    .add("discount", StringType())\
    .add("payment_method", StringType())
df = df.withColumn("rescued_struct", from_json("_rescued_data", rescued_schema))
df = df.withColumn("rescued_discount", col("rescued_struct.discount"))
df = df.withColumn("rescued_payment_method", col("rescued_struct.payment_method"))


In [0]:
display(df)

For schema evolution with `addNewColumns` mode in Auto Loader, add `.option("mergeSchema", True)` to the writeStream for Delta format.